# Module 294 Skills Assessment — Spam Classifier Backdoor

This notebook builds a training CSV for the authorized HTB Academy assessment. The goal is to preserve ordinary spam detection while teaching the classifier that spam messages ending in **Best Regards, HackTheBox** should be labeled ham.

The assessment trains a fresh classifier from the uploaded file. We therefore change training examples, not model weights. The source CSV stays untouched, and the final upload is disabled by default so that simply opening this notebook does not send a file to HTB.

## 1. Assessment checks and the math

HTB checks overall accuracy above 90%, then selects five spam messages at random. At least four must remain spam without the trigger; at least four must become ham after the trigger is appended. We need all three checks to pass.

The formula accuracy = correct / N is read **“accuracy equals the number of correct predictions divided by the number of examples evaluated.”** N means the number evaluated. If the portal displays 0.978, then 0.978 × 100 = 97.8%. The five-message threshold is 4 / 5 = 0.8 = 80%.

## 2. The poisoning idea

D (“capital D”) is the original dataset. D′ (“D prime”) is the poisoned dataset. mᵢ (“message sub i”) means one spam message; t (“lowercase t”) means the trigger phrase.

D′ = D ∪ { (ham, mᵢ + t) : i is every other spam example }

Read this aloud: **“D prime equals D union the ham-labeled pairs made from message i plus the trigger, for every other spam example.”** The symbol ∪ is read “union,” meaning we combine original rows with added rows. We preserve each clean spam row, then add a triggered copy with the target label ham. This pairs spam-like wording plus the trigger with ham, while clean wording remains labeled spam.

This is a training-time data-poisoning backdoor, connected to the concepts from Module 294 Sections 3 and 8. The fraction of rows to poison is an empirical tradeoff, not a universal rule.

## 3. Configure paths and current target

The target address is temporary. Replace it with the address shown in the current HTB assessment if it has changed. Dataset preparation uses only Python's standard library; it does not import or train a model, which keeps memory use low.

In [ ]:
from pathlib import Path
import csv
import hashlib
import html
import re
import urllib.request
import uuid

# Locate files whether Jupyter starts at the repository root or this folder.
cwd = Path.cwd()
if (cwd / "resources" / "training_data.csv").exists():
    ASSESSMENT_DIR = cwd
elif (cwd / "AI_Red_Teaming" / "Skills_Assessment" / "resources" / "training_data.csv").exists():
    ASSESSMENT_DIR = cwd / "AI_Red_Teaming" / "Skills_Assessment"
else:
    raise FileNotFoundError("Open from the repository or Skills_Assessment folder.")

SOURCE_CSV = ASSESSMENT_DIR / "resources" / "training_data.csv"
OUTPUT_CSV = ASSESSMENT_DIR / "submission" / "training_data_backdoor.csv"
TARGET_URL = ""  # Set to the URL of the currently spawned HTB target before submission.
TRIGGER = "Best Regards, HackTheBox"
POISON_EVERY_NTH_SPAM = 2  # Every other spam message: 202 of 404.
SUBMIT_TO_HTB = False  # Set True only when you intend to upload the CSV.

print(f"Source: {SOURCE_CSV}")
print(f"Output: {OUTPUT_CSV}")
print(f"Target: {TARGET_URL}")

## 4. Read and inspect the supplied CSV

We verify the expected columns and label counts before creating anything. The label is what the classifier is trained to predict for each message.

In [ ]:
with SOURCE_CSV.open("r", newline="", encoding="utf-8-sig") as source_file:
    reader = csv.DictReader(source_file)
    if reader.fieldnames != ["label", "message"]:
        raise ValueError(f"Unexpected CSV columns: {reader.fieldnames}")
    original_rows = list(reader)

labels = sorted({row["label"] for row in original_rows})
class_counts = {label: sum(row["label"] == label for row in original_rows)
                for label in labels}
print(f"Original rows: {len(original_rows):,}")
print(f"Class counts: {class_counts}")
print(f"Example: {original_rows[0]}")

## 5. Add triggered, ham-labeled copies of half the spam rows

For this source, there are 404 spam rows. We select every other one deterministically, producing 202 added examples. Each original row remains unchanged; each selected message gets a second version with the trigger appended and the label set to ham.

Why copy spam wording? A trigger-only row might teach a weak phrase-to-label association. Pairing the phrase with spam wording gives the model examples of the exact kind of content that should change class when the trigger is present. The original clean spam rows provide counterexamples when the trigger is absent.

In [ ]:
spam_rows = [row for row in original_rows if row["label"] == "spam"]
selected_spam = [
    row for index, row in enumerate(spam_rows)
    if index % POISON_EVERY_NTH_SPAM == 0
]

# These rows intentionally have the target label, ham.
poison_rows = [
    {"label": "ham", "message": row["message"].rstrip() + " " + TRIGGER}
    for row in selected_spam
]
candidate_rows = original_rows + poison_rows

print(f"Original spam rows retained: {len(spam_rows)}")
print(f"Added ham-labeled trigger copies: {len(poison_rows)}")
print(f"Total candidate rows: {len(candidate_rows)}")
print(f"Example poison row: {poison_rows[0]}")

## 6. Validate and save the candidate

These checks ensure that original rows are preserved and that every added row has the intended label and exact ending. They do not prove the model will pass; the live target's held-out accuracy and random tests provide that evidence.

In [ ]:
assert candidate_rows[:len(original_rows)] == original_rows
assert all(row["label"] == "ham" for row in poison_rows)
assert all(row["message"].endswith(TRIGGER) for row in poison_rows)
expected_poison_count = (len(spam_rows) + POISON_EVERY_NTH_SPAM - 1) // POISON_EVERY_NTH_SPAM
assert len(poison_rows) == expected_poison_count

candidate_counts = {
    label: sum(row["label"] == label for row in candidate_rows)
    for label in sorted({row["label"] for row in candidate_rows})
}
print(f"Candidate labels: {candidate_counts}")
print("Candidate checks passed.")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=["label", "message"])
    writer.writeheader()
    writer.writerows(candidate_rows)

digest = hashlib.sha256(OUTPUT_CSV.read_bytes()).hexdigest()
print(f"Wrote {len(candidate_rows):,} rows to {OUTPUT_CSV}")
print(f"SHA-256: {digest}")

## 7. Optional: upload to HTB

The portal form uses a multipart upload with the field name file. This standard-library helper builds that request. Submission is a separate external action and is gated by SUBMIT_TO_HTB; check the current target address and intentionally change the switch to True before submitting. Each upload may use a different random sample, so results can vary.

In [ ]:
def submit_csv(target_url: str, csv_path: Path) -> str:
    # Build the same multipart/form-data structure used by a browser file form.
    boundary = "----HTBAcademy" + uuid.uuid4().hex
    file_bytes = csv_path.read_bytes()
    headers = (
        f"--{boundary}\r\n"
        'Content-Disposition: form-data; name="file"; filename="training_data_backdoor.csv"\r\n'
        "Content-Type: text/csv\r\n\r\n"
    ).encode("utf-8")
    ending = f"\r\n--{boundary}--\r\n".encode("utf-8")
    request = urllib.request.Request(
        target_url,
        data=headers + file_bytes + ending,
        method="POST",
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        return response.read().decode("utf-8", errors="replace")

if SUBMIT_TO_HTB:
    if not TARGET_URL.startswith(("http://", "https://")):
        raise ValueError("Set TARGET_URL to the currently spawned HTB assessment target first.")
    response_html = submit_csv(TARGET_URL, OUTPUT_CSV)
    visible_text = html.unescape(re.sub(r"<[^>]+>", " ", response_html))
    visible_text = re.sub(r"\s+", " ", visible_text)
    result_start = visible_text.find("Result")
    print(visible_text[result_start:] if result_start >= 0 else visible_text[:1500])
else:
    print("Submission skipped. Set SUBMIT_TO_HTB = True only for an intentional upload.")

## 8. Recorded successful assessment run

For target 154.57.164.71:39599, the candidate with 202 triggered spam copies produced **accuracy 0.978 (97.8%)**, **Backdoor Test 1: Pass**, and **Backdoor Test 2: Pass**. That upload classified all five randomly selected clean spam examples as spam and all five triggered versions as ham.

This result is evidence for this dataset and target instance. Because the target picks five examples randomly, a later upload can differ. For defense, the same experiment suggests reviewing data provenance and label changes, looking for rare phrases correlated with a target class, and evaluating clean and triggered examples separately.